**1.Setup and Get Data**

1.1 Import Dependencies and Setup

In [ ]:
import os
import time
import uuid
import cv2

1.2 Collect Images Using OpenCV

In [ ]:
uuid.uuid1()

In [24]:
IMAGES_PATH = os.path.join('\\FaceDetection\\data\\images','images')
number_images = 30

In [ ]:
cap = cv2.VideoCapture(0)
for imgnum in range(number_images):
    print('Collecting image {}'.format((imgnum)))
    ret, frame = cap.read()
    if not ret:
        print("Can't open camera")
        break
    imgname = os.path.join(IMAGES_PATH,f'{str(uuid.uuid1())}.jpg')
    cv2.imwrite(imgname,frame)
    cv2.imshow(('frame'),frame)
    time.sleep(0.5)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

1.3 Annotate Images with LabelMe

In [ ]:
!labelme

 **2. Review Dataset and Build Image Loading Funtion**


2.1 import TF and Deps

In [ ]:
import tensorflow as tf
import cv2
import json
import numpy as np
from matplotlib import  pyplot as plt

2.2 Limit GPU Memory Growth

In [ ]:
#Avoid OOM errors by setting GPU Memory Consumption Growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus :
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
print(tf.config.list_physical_devices())

2.3 Load Image into TF Data Pipeline

In [ ]:
# images = tf.data.Dataset.list_files('/content/data/images/*.jpg',shuffle=False)
images = tf.data.Dataset.list_files('\\FaceDetection\\data\\images\\*.jpg',shuffle=True)
# images = images.shuffle(buffer_size=100)


In [ ]:
def load_image(x):
  byte_img = tf.io.read_file(x)
  img = tf.io.decode_jpeg(byte_img)
  return img

In [ ]:
images = images.map(load_image)

In [ ]:
def preprocess(img):
    img = tf.image.resize(img, (224, 224))
    img = tf.cast(img, tf.float32)  / 255
    return img

images = images.map(preprocess)

In [ ]:
for img in images.take(5):
    print(img.shape)
    print(img.numpy().min(), img.numpy().max())

In [ ]:
images.as_numpy_iterator().next()

2.4 View Raw Images with Matplotlib

In [ ]:
image_generator = images.batch(4).as_numpy_iterator()

In [ ]:
plot_images = image_generator.next()

In [ ]:
fig, ax = plt.subplots(ncols = 4,figsize=(20,20))
for idx, image in enumerate(plot_images):
    ax[idx].imshow(image)
plt.show()

**3.Partition Unaugmented Data**


3.1 MANUALLY SPLIT DATA INTO TRAIN TEST AND VAL

3.2 Move the Matching Labels

In [ ]:
for folder in ['train','test','val']:
    for file in os.listdir(os.path.join('\\Project\\FaceDetection\\data', folder, 'images')):
        filename = file.split('.')[0]+'.json'
        existing_filepath = os.path.join('\\Project\\FaceDetection\\data','label', filename)
        if os.path.exists(existing_filepath):
            new_filepath = os.path.join('\\Project\\FaceDetection\\data',folder,'label',filename)
            os.replace(existing_filepath, new_filepath)

**4. Apply Image Augmentation on Images and Labels using Albumentations**


In [ ]:
img = cv2.imread(os.path.join('\\Project\\FaceDetection\\data','train','images','0635f4f1-39b9-11f1-a32f-48e7daa73ab0.jpg'))

In [ ]:
img.shape

4.1 Setup Albumentations Transform Pipeline

In [ ]:
import albumentations as alb

In [ ]:
augmentor = alb.Compose(
    [
        # alb.RandomCrop(width = 450, height =450),
        # alb.AtLeastOneBboxRandomCrop(width=450, height=450),
        alb.HorizontalFlip(p=0.5),
        alb.RandomBrightnessContrast(p=0.2),
        alb.RandomGamma(p=0.2),
        alb.RGBShift(p=0.2),
        # alb.VerticalFlip(p=0.5),
        ],
        bbox_params = alb.BboxParams(format='albumentations',
                                      label_fields=['class_labels'],)
                                      # min_visibility=0.4)
)

 4.2 Load a Test Image and Annotation with OpenCV and JSON

In [ ]:
img = cv2.imread(os.path.join('\\FaceDetection\\data','train','images','0635f4f1-39b9-11f1-a32f-48e7daa73ab0.jpg'))

In [ ]:
with open(os.path.join('\\FaceDetection\\data','train','label','0635f4f1-39b9-11f1-a32f-48e7daa73ab0.json'),'r') as f :
  label = json.load(f)

In [ ]:
label['shapes'][0]['points']

4.3 Extract Coordiantes and Rescale to Match Image Resolution

In [ ]:
coords = [0,0,0,0]
coords[0] = label['shapes'][0]['points'][0][0]
coords[1] = label['shapes'][0]['points'][0][1]
coords[2] = label['shapes'][0]['points'][1][0]
coords[3] = label['shapes'][0]['points'][1][1]

In [ ]:
coords

In [ ]:
h, w = img.shape[:2]

coords = np.array(coords)
coords = coords / np.array([w, h, w, h])
coords = coords.tolist()
coords = [max(0, min(1, c)) for c in coords]

In [ ]:
coords

4.4 Apply Augmentation and View Results

In [ ]:
augmented = augmentor(image=img, bboxes=[coords], class_labels=['face'])

In [ ]:
augmented['bboxes'][0][2:]

In [ ]:
augmented['bboxes']

In [ ]:
if len(augmented['bboxes']) > 0:
    x_min, y_min, x_max, y_max = map(int, augmented['bboxes'][0])
    cv2.rectangle(augmented['image'],
                  (int(x_min), int(y_min)),
                  (int(x_max), int(y_max)),
                  (255, 0, 0), 2)
    plt.imshow(augmented['image'])
else:
    print("Bounding box crop outside the picture, try augment again!")
    plt.imshow(augmented['image'])
plt.show()

**5. Build and Run Augmentation**


5.1 Run Augmentation Pipeline


In [ ]:
path = os.path.join('\\FaceDetection\\data\\images\\aug_data', partition, 'label', f'{image.split(".")[0]}.{x}.json')
print("Saving to:", path)


In [ ]:
base_dir = '\\FaceDetection\\data\\images\\aug_data'

for partition in ['train','test','val']:
    input_img_dir = os.path.join('\\FaceDetection\\data', partition, 'images')
    if not os.path.exists(input_img_dir):
        print(f"Skipping {partition}: {input_img_dir} does not exist.")
        continue

    print(f"Processing {partition} images...")
    for image in os.listdir(input_img_dir):
        img = cv2.imread(os.path.join(input_img_dir, image))
        if img is None:
            print(f"Could not read image: {image}")
            continue

        coords = [0,0,0.00001,0.00001]
        label_path = os.path.join('\\FaceDetection\\data', partition, 'label', f'{image.split(".")[0]}.json')

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                label = json.load(f)

            if len(label['shapes']) > 0:
                coords[0] = label['shapes'][0]['points'][0][0]
                coords[1] = label['shapes'][0]['points'][0][1]
                coords[2] = label['shapes'][0]['points'][1][0]
                coords[3] = label['shapes'][0]['points'][1][1]

                h, w = img.shape[:2]
                coords = list(np.divide(coords, [w, h, w, h]))
                coords = [max(0.0, min(1.0, c)) for c in coords]

        try:
            img_dir = os.path.join(base_dir, partition, 'images')
            label_dir = os.path.join(base_dir, partition, 'label')
            os.makedirs(img_dir, exist_ok=True)
            os.makedirs(label_dir, exist_ok=True)

            for x in range(60):
                augmented = augmentor(image=img, bboxes=[coords], class_labels=['face'])

                img_filename = f'{image.split(".")[0]}.{x}.jpg'
                cv2.imwrite(os.path.join(img_dir, img_filename), augmented['image'])

                annotation = {'image': image}
                if os.path.exists(label_path) and len(augmented['bboxes']) > 0:
                    annotation['bbox'] = augmented['bboxes'][0]
                    annotation['class'] = 1
                else:
                    annotation['bbox'] = [0,0,0,0]
                    annotation['class'] = 0

                with open(os.path.join(label_dir, f'{image.split(".")[0]}.{x}.json'), 'w') as f:
                    json.dump(annotation, f)

            print(f"Finished 60 augmentations for {image}")

        except Exception as e:
            print(f"Error processing {image}: {e}")

5.2 Load Augmented Images to Tensorflow Dataset

In [ ]:
train_images = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\train\\images\\*.jpg', shuffle=False)
train_images = train_images.map(load_image)
train_images = train_images.map(lambda x: tf.image.resize(x, (120,120)))
train_images = train_images.map(lambda x: x/255)

In [ ]:
test_images = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\test\\images\\*.jpg', shuffle=False)
test_images = test_images.map(load_image)
test_images = test_images.map(lambda x: tf.image.resize(x, (120,120)))
test_images = test_images.map(lambda x: x/255)

In [ ]:

val_images = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\val\\images\\*.jpg', shuffle=False)
val_images = val_images.map(load_image)
val_images = val_images.map(lambda x: tf.image.resize(x, (120,120)))
val_images = val_images.map(lambda x: x/255)

In [ ]:
train_images.as_numpy_iterator().next()

**6. Prepare Labels**


6.1 Build Label Loading Function


In [ ]:
def load_labels(label_path):
  with open(label_path.numpy(), 'r', encoding = "utf-8") as f:
    label = json.load(f)

  return [label['class']], label['bbox']

6.2 Load Labels to TensorFlow Dataset


In [ ]:
train_labels = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\train\\label\\*.json', shuffle=False)
train_labels = train_labels.map(lambda x : tf.py_function(load_labels, [x], Tout=[tf.uint8, tf.float16]))

In [ ]:
test_labels = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\test\\label\\*.json', shuffle=False)
test_labels = test_labels.map(lambda x : tf.py_function(load_labels, [x], Tout=[tf.uint8, tf.float16]))

In [ ]:
val_labels = tf.data.Dataset.list_files('\\FaceDetection\\aug_data\\val\\label\\*.json', shuffle=False)
val_labels = val_labels.map(lambda x : tf.py_function(load_labels, [x], Tout=[tf.uint8, tf.float16]))

**7. Combine Label and Image Samples**

7.1 Check Partition Length

In [ ]:
len(train_images),len(train_labels),len(test_images),len(test_labels),len(val_images),len(val_labels)

7.2 Create Fianl Datasets(Images/Labels)

In [ ]:
train = tf.data.Dataset.zip((train_images, train_labels))
train = train.shuffle(1000)
train = train.batch(8)
train = train.prefetch(4)

In [ ]:
test = tf.data.Dataset.zip((test_images, test_labels))
test = test.shuffle(1300)
test = test.batch(8)
test = test.prefetch(4)

In [ ]:
val = tf.data.Dataset.zip((val_images, val_labels))
val = val.shuffle(1000)
val = val.batch(8)
val = val.prefetch(4)

In [ ]:
train = tf.data.Dataset.zip((train_images, train_labels))
train = train.cache()
train = train.shuffle(1000)
train = train.batch(8)
train = train.prefetch(4)

In [ ]:
train.as_numpy_iterator().next()[1]

7.3 View Images and Annotations

In [ ]:
data_samples  = train.as_numpy_iterator()

In [ ]:
res = data_samples.next()

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(20,20))

for idx in range(4):
    sample_image = res[0][idx].copy()
    sample_coords = res[1][1][idx]

    # sample_image = (sample_image * 255).astype(np.uint8)

    cv2.rectangle(
        sample_image,
        tuple(np.multiply(sample_coords[:2], [120,120]).astype(int)),
        tuple(np.multiply(sample_coords[2:], [120,120]).astype(int)),
        (255, 0, 0),
        2
    )

    # sample_image = cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB)

    ax[idx].imshow(sample_image)
    # ax[idx].axis('off')

In [ ]:
sample_image  = res[1][1].copy()
print(sample_image.flags.writeable)

**8.Build Depp Learning using the Functional API**

8.1 Import Layers and Base Network

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Dense, GlobalMaxPooling2D
from tensorflow.keras.applications import VGG16

8.2 Download VGG16

In [ ]:
vgg = VGG16(include_top=False)

In [ ]:
vgg.summary()

8.3 Build instance of Network

In [ ]:
def build_model() :
  input_layer = Input(shape=(120,120,3))

  vgg = VGG16(include_top=False)(input_layer)
  #classification model
  f = GlobalMaxPooling2D()(vgg)
  class1 = Dense(2048, activation='relu')(f)
  class2 = Dense(1, activation='sigmoid')(class1)

  #bounding box model
  regress1 = Dense(2048, activation='relu')(f)
  regress2 = Dense(4, activation='sigmoid')(regress1)

  facetracker = Model(inputs=input_layer, outputs=[class2, regress2])
  return facetracker

In [ ]:
def build_model() :
  input_layer = Input(shape=(120,120,3))

  vgg = VGG16(include_top=False)(input_layer)
  #classification model
  f1 = GlobalMaxPooling2D()(vgg)
  class1 = Dense(2048, activation='relu')(f1)
  class2 = Dense(1, activation='sigmoid')(class1)

  #bounding box model
  f2 = GlobalMaxPooling2D()(vgg)
  regress1 = Dense(2048, activation='relu')(f2)
  regress2 = Dense(4, activation='sigmoid')(regress1)

  facetracker = Model(inputs=input_layer, outputs=[class2, regress2])
  return facetracker

8.4 Test out Neural Network

In [ ]:
facetracker = build_model()

In [ ]:
facetracker.summary()

In [ ]:
X, y = train.as_numpy_iterator().next()

In [ ]:
X.shape

In [ ]:
classes, coords = facetracker.predict(X)

In [ ]:
classes, coords

**9. Define Losses and Optimizers**


9.1 Define Optimizer and LR

In [ ]:
batches_per_epoch = len(train)
lr_delay = (1/0.75 -1)/batches_per_epoch

In [ ]:
opt =  tf.keras.optimizers.Adam(learning_rate=0.0001, decay=lr_delay)

9.2 Create Localization Loss and Classification Loss

In [ ]:
def localization_loss(y_true, yhat):
    delta_coord = tf.reduce_sum(tf.square(y_true[:,:2] - yhat[:,:2]))

    h_true = y_true[:,3] - y_true[:,1]
    w_true = y_true[:,2] - y_true[:,0]

    h_pred = yhat[:,3] - yhat[:,1]
    w_pred = yhat[:,2] - yhat[:,0]

    delta_size = tf.reduce_sum(tf.square(w_true - w_pred) + tf.square(h_true-h_pred))

    return delta_coord + delta_size

In [ ]:
classloss = tf.keras.losses.BinaryCrossentropy()
regressloss = localization_loss

In [ ]:
classloss

In [ ]:
regressloss

9.3 Test out Loss Metrics

In [ ]:
localization_loss(y[1],coords)

In [ ]:
classloss(y[0],classes)

In [ ]:
regressloss(y[1],coords)

**10. Train Neural Network**


10.1 Create Custom Model Class

In [ ]:
class FaceTracker(Model):
    def __init__(self, facetracker, **kwargs):
        super().__init__(**kwargs)
        self.model = facetracker

    def compile(self, opt, classloss, localizationloss, **kwargs):
        super().compile(**kwargs)
        self.closs = classloss
        self.lloss = localizationloss
        self.opt = opt

    def train_step(self, batch):
        X, y = batch

        with tf.GradientTape() as tape:
            classes, coords = self.model(X, training=True)

            y_class = tf.cast(y[0], tf.float32)
            y_regress = tf.cast(y[1], tf.float32)

            y_class = tf.reshape(y_class, [-1, 1])
            y_regress = tf.reshape(y_regress, [-1, 4])

            batch_classloss = self.closs(y_class, classes)
            batch_localizationloss = self.lloss(y_regress, coords)

            total_loss = batch_localizationloss + 0.5 * batch_classloss

            grad = tape.gradient(total_loss, self.model.trainable_variables)

        self.opt.apply_gradients(zip(grad, self.model.trainable_variables))

        return {"total_loss": total_loss, "class_loss": batch_classloss, "regress_loss": batch_localizationloss}

    def test_step(self, batch):
        X, y = batch

        classes, coords = self.model(X, training=False)

        y_class = tf.cast(y[0], tf.float32)
        y_regress = tf.cast(y[1], tf.float32)

        y_class = tf.reshape(y_class, [-1, 1])
        y_regress = tf.reshape(y_regress, [-1, 4])

        batch_classloss = self.closs(y_class, classes)
        batch_localizationloss = self.lloss(y_regress, coords)

        total_loss = batch_localizationloss + 0.5 * batch_classloss

        return {"total_loss": total_loss, "class_loss": batch_classloss, "regress_loss": batch_localizationloss}

    def call(self, X, **kwargs):
        return self.model(X, **kwargs)

In [ ]:
model = FaceTracker(facetracker)

In [ ]:
model.compile(opt,classloss,regressloss)

10.2 Train

In [34]:
logdir = '\\FaceDetection\\logs'

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

In [ ]:
hist = model.fit(train, epochs=10, validation_data=val, callbacks=[tensorboard_callback])

10.3 Plot Performance

In [ ]:
hist.history

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(20,5))

ax[0].plot(hist.history['total_loss'], color='teal', label='loss')
ax[0].plot(hist.history['val_total_loss'], color='orange', label='val loss')
ax[0].title.set_text('Loss')
ax[0].legend()

ax[1].plot(hist.history['class_loss'], color='teal', label='class loss')
ax[1].plot(hist.history['val_class_loss'], color='orange', label='val class loss')
ax[1].title.set_text('Classification Loss')
ax[1].legend()

ax[2].plot(hist.history['regress_loss'], color='teal', label='regress loss')
ax[2].plot(hist.history['val_regress_loss'], color='orange', label='val regress loss')
ax[2].title.set_text('Regression Loss')
ax[2].legend()

plt.show()

**11. Make Predictions**

11.1 Make Prediction on Test set

In [ ]:
test_data = test.as_numpy_iterator()

In [ ]:
test_sample = test_data.next()

In [ ]:
yhat = facetracker.predict(test_sample[0])

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(20,20))
for idx in range(4):
    sample_image = test_sample[0][idx].copy()
    sample_coords = yhat[1][idx]

    if yhat[0][idx] > 0.9:
        cv2.rectangle(sample_image,
                      tuple(np.multiply(sample_coords[:2], [120,120]).astype(int)),
                      tuple(np.multiply(sample_coords[2:], [120,120]).astype(int)),
                            (255,0,0), 2)

    ax[idx].imshow(sample_image)

11.2 Save the Model

In [ ]:
from tensorflow.keras.models import load_model

In [27]:
facetracker.save('\\FaceDetection\\models\\facetracker.h5')

In [33]:
facetracker = load_model('\\FaceDetection\\models\\facetracker.h5')

11.3 Realtime Detection

In [ ]:
import cv2

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [26]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# load model
facetracker = load_model("models\\facetracker.h5")

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Can't open camera")
    exit()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # convert màu + resize
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = tf.image.resize(rgb, (120,120)).numpy()

    # predict
    yhat = facetracker.predict(np.expand_dims(resized/255,0), verbose=0)
    sample_coords = yhat[1][0]

    # vẽ khi có face
    if yhat[0][0] > 0.5:
        h, w, _ = frame.shape

        # bbox chính
        start = tuple(np.multiply(sample_coords[:2], [w, h]).astype(int))
        end   = tuple(np.multiply(sample_coords[2:], [w, h]).astype(int))

        cv2.rectangle(frame, start, end, (255,0,0), 2)

        # label box
        label_start = (start[0], start[1] - 30)
        label_end   = (start[0] + 80, start[1])

        cv2.rectangle(frame, label_start, label_end, (255,0,0), -1)

        # text
        cv2.putText(frame, 'face',
                    (start[0], start[1] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (255,255,255), 2, cv2.LINE_AA)

    cv2.imshow('EyeTrack', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()